# Phase 11: Sales Forecasting (Prophet)

Aggregates transactions to daily revenue and fits Prophet with weekly and yearly seasonality, backtesting on the most recent 60 days before producing a 90-day forward forecast.

In [3]:
%pip install prophet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup

In [4]:
import pandas as pd, numpy as np, joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv('../data/polokwane_sales_clean.csv', parse_dates=['date'])
df = df[df['value_zar'] > 0]

daily = df.groupby('date', as_index=False)['value_zar'].sum().rename(columns={'date': 'ds', 'value_zar': 'y'})
daily = daily.sort_values('ds').reset_index(drop=True)
print("Daily series length:", len(daily))
print(daily.head())
print(daily.tail())

# Train/test split on time series: last 60 days held out
horizon = 60
train = daily.iloc[:-horizon]
test = daily.iloc[-horizon:]

m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
m.fit(train)

future = m.make_future_dataframe(periods=horizon)
forecast = m.predict(future)

test_forecast = forecast.set_index('ds').loc[test['ds']]
mae = mean_absolute_error(test['y'], test_forecast['yhat'])
rmse = np.sqrt(mean_squared_error(test['y'], test_forecast['yhat']))
mape = np.mean(np.abs((test['y'].values - test_forecast['yhat'].values) / test['y'].values)) * 100

print(f"\nProphet forecast holdout ({horizon} days):")
print(f"  MAE:  R{mae:,.2f}")
print(f"  RMSE: R{rmse:,.2f}")
print(f"  MAPE: {mape:.1f}%")

# Refit on full data for the deployed forecast used in the app
m_full = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
m_full.fit(daily)
future_full = m_full.make_future_dataframe(periods=90)
forecast_full = m_full.predict(future_full)

fig = m_full.plot(forecast_full)
plt.title('Daily Sales Revenue Forecast (90-day horizon) — Prophet')
plt.xlabel('Date')
plt.ylabel('Revenue (ZAR)')
plt.tight_layout()
plt.savefig('../outputs/prophet_forecast.png', dpi=150)
plt.close()

fig2 = m_full.plot_components(forecast_full)
plt.tight_layout()
plt.savefig('../outputs/prophet_components.png', dpi=150)
plt.close()

joblib.dump(m_full, '../models/prophet_model.joblib')
daily.to_csv('../data/daily_sales.csv', index=False)
forecast_full[['ds','yhat','yhat_lower','yhat_upper']].to_csv('../data/forecast_90d.csv', index=False)

with open('../outputs/prophet_metrics.json', 'w') as f:
    import json
    json.dump({'mae': mae, 'rmse': rmse, 'mape': mape, 'holdout_days': horizon}, f, indent=2)

print("\nSaved prophet_model.joblib, forecast plots, daily_sales.csv, forecast_90d.csv")

c:\Users\vulev\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.
01:33:31 - cmdstanpy - INFO - Chain [1] start processing


Daily series length: 1049
          ds          y
0 2023-07-01  130730.55
1 2023-07-02   26821.88
2 2023-07-03  106960.31
3 2023-07-04   93543.94
4 2023-07-05   99810.64
             ds          y
1044 2026-08-21  103083.17
1045 2026-08-22   93890.84
1046 2026-08-23   25823.52
1047 2026-08-24   85914.97
1048 2026-08-25  108439.97


01:33:32 - cmdstanpy - INFO - Chain [1] done processing
01:33:32 - cmdstanpy - INFO - Chain [1] start processing
01:33:32 - cmdstanpy - INFO - Chain [1] done processing



Prophet forecast holdout (60 days):
  MAE:  R19,485.39
  RMSE: R24,954.64
  MAPE: 24.4%

Saved prophet_model.joblib, forecast plots, daily_sales.csv, forecast_90d.csv
